# Part 1 — Hourly Electricity Supply by Source (Germany)

本 Notebook 专门回答题目第一问：

> Plot the time series of hourly electricity supply by source. Discuss trends, seasonality, and data issues, if any.

数据处理口径与预测 Notebook 保持一致：
- `generation_series.csv`：小时宽表，`reference_time_start` + 各 `curve_id`
- `curve_metadata.csv`：`curve_id` 与 `Technology_is`、`Technology_Fuel_is` 的映射
- 按技术类别聚合子类发电量，单位沿用原始数据（MW）
- 时间戳保留 UTC 作为索引；分析德国日内/星期周期时转换为 `Europe/Berlin`

## 本 Notebook 的图表结构

1. 各能源类别小时级时间序列：观察长期趋势、结构变化和异常。
2. ACF：观察小时惯性、日内（24h）、周内（168h）和年度（8760h）依赖。
3. Source correlation heatmap：比较能源类别之间的同期线性关系。
4. Month × Hour heatmap：重点展示 Solar 的强日内—季节交互，也辅助观察 Thermal。
5. Renewable quintile displacement plot：观察 Wind + Solar 供给水平不同的时候，Thermal 平均出力如何变化。

最后两张图分别对应此前分析中发现的 Solar `Month × Hour` 周期解释度较高（约 0.846），以及 Thermal 随 Wind + Solar 供给增加而下降的 merit-order 替代关系。数值是此前截图中报告的结果，不代表本 Notebook 尚未运行的数据结果。

## 0. Imports and file paths

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.graphics.tsaplots import plot_acf

DATA_DIR = Path(".")
SERIES_PATH = DATA_DIR / "generation_series.csv"
META_PATH = DATA_DIR / "curve_metadata.csv"

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

## 1. Load hourly series and metadata

In [ ]:
series = pd.read_csv(SERIES_PATH)
meta = pd.read_csv(META_PATH)

series.columns = series.columns.astype(str).str.strip()
meta.columns = meta.columns.astype(str).str.strip()

time_col = "reference_time_start"
required_meta = ["curve_id", "Technology_is", "Technology_Fuel_is"]
missing_meta = [c for c in required_meta if c not in meta.columns]
if missing_meta:
    raise ValueError(f"Metadata missing columns: {missing_meta}")
if time_col not in series.columns:
    raise ValueError(f"Hourly data missing timestamp column: {time_col}")

series[time_col] = pd.to_datetime(series[time_col], utc=True, errors="coerce")
bad_time_rows = series[time_col].isna().sum()
duplicate_time_rows = series[time_col].dropna().duplicated().sum()
series = (series.dropna(subset=[time_col])
                .sort_values(time_col)
                .drop_duplicates(time_col, keep="last")
                .set_index(time_col))

meta["curve_id"] = meta["curve_id"].astype(str).str.strip()
meta["Technology_is"] = meta["Technology_is"].astype("string").str.strip()
meta["Technology_Fuel_is"] = meta["Technology_Fuel_is"].astype("string").str.strip()

matched = [c for c in series.columns if c in set(meta["curve_id"])]
if not matched:
    raise ValueError("No curve_id in metadata matches columns in generation_series.csv.")

series[matched] = series[matched].apply(pd.to_numeric, errors="coerce")

print("Raw wide data shape:", series.shape)
print("Matched curve columns:", len(matched))
print("Invalid timestamp rows removed:", bad_time_rows)
print("Time range:", series.index.min(), "to", series.index.max())
display(meta[meta["curve_id"].isin(matched)][required_meta].head(25))

## 2. Map technology labels and aggregate subcategories

In [ ]:
def map_source(technology):
    x = "" if pd.isna(technology) else str(technology).lower().strip()

    # Exclude national aggregate rows to avoid double counting.
    if "power generation" in x:
        return "Aggregate"
    if "thermal power" in x:
        return "Thermal"
    if "wind turbines" in x or "wind" in x:
        return "Wind"
    if "solar" in x:
        return "Solar"
    if "pump storage" in x or "pumped storage" in x:
        return "PumpedStorage"
    if "nuclear" in x:
        return "Nuclear"
    if any(k in x for k in ["reservoir", "run of river", "hydro"]):
        return "Hydro"
    if "geothermal" in x:
        return "Geothermal"
    return "Other"

meta["source"] = meta["Technology_is"].map(map_source)

# Regular hourly UTC index makes missing hours visible.
full_index = pd.date_range(series.index.min(), series.index.max(), freq="h", tz="UTC")
series = series.reindex(full_index)
series.index.name = time_col

source_series = {}
source_order = [
    "Thermal", "Wind", "Solar", "Hydro", "Nuclear",
    "PumpedStorage", "Geothermal", "Other"
]

for source in source_order:
    ids = meta.loc[meta["source"].eq(source), "curve_id"].tolist()
    ids = [c for c in ids if c in series.columns]
    if ids:
        # min_count=1 preserves NaN when all subcategory observations are missing.
        source_series[source] = series[ids].sum(axis=1, min_count=1)

hourly = pd.DataFrame(source_series)
print("Aggregated hourly data:", hourly.shape)
display(hourly.head())
display(meta.groupby("source", dropna=False).agg(
    n_curves=("curve_id", "nunique"),
    technologies=("Technology_is", lambda x: ", ".join(sorted(set(x.dropna().astype(str)))))
))

## 3. Data quality checks

先检查数据是否适合做趋势和周期分析：

- 时间戳重复、缺失或不规则；
- 每个能源类别缺失率；
- 是否存在负发电量；
- 缺失子类被聚合时可能造成总量低估。

注意：`sum(min_count=1)` 可以避免“所有子类都缺失却被误记为 0”，但如果一个类别只有部分子类缺失，求和仍会得到已观测子类之和。因此需要同时检查原始曲线的缺失率。

In [ ]:
# Raw curve-level missingness, before aggregation.
curve_missing = series[matched].isna().mean().sort_values(ascending=False)
print("Most-missing raw curves:")
display(curve_missing.head(15).rename("missing_rate").to_frame())

# Aggregated source-level diagnostics.
quality = pd.DataFrame({
    "Missing_Rate": hourly.isna().mean(),
    "Mean_MW": hourly.mean(),
    "Std_MW": hourly.std(),
    "Min_MW": hourly.min(),
    "Max_MW": hourly.max(),
    "Negative_Count": (hourly < 0).sum()
}).sort_values("Missing_Rate", ascending=False)

display(quality)
print("Duplicate timestamp rows removed during load:", duplicate_time_rows)
print("Expected hourly rows:", len(pd.date_range(hourly.index.min(), hourly.index.max(), freq="h", tz="UTC")))
print("Actual hourly rows:", len(hourly))

## 4. Plot hourly time series by source

In [ ]:
plot_sources = [c for c in source_order if c in hourly.columns]
fig, axes = plt.subplots(len(plot_sources), 1, figsize=(16, 2.7 * len(plot_sources)), sharex=True)
if len(plot_sources) == 1:
    axes = [axes]

for ax, source in zip(axes, plot_sources):
    ax.plot(hourly.index, hourly[source], lw=0.45)
    ax.set_title(f"{source} hourly generation")
    ax.set_ylabel("MW")

axes[-1].set_xlabel("Time (UTC)")
plt.tight_layout()
plt.show()

### How to discuss the time-series plots

按图回答三件事：

1. **Trend / level shifts：** 长期平均出力是否上升、下降或发生结构性变化？是否存在明显的阶段性水平变化？
2. **Seasonality：** 是否存在重复的日内、星期或年度波动？Solar 通常最明显地呈现日出—日落周期及季节变化；Wind 的季节变化可能明显，但日内规律不一定强。
3. **Data issues：** 是否有长时间缺口、突然归零、异常尖峰或不连续区间？结合上一节缺失率表判断，不能把缺失值直接解释为零发电。

从此前数据分析截图可先形成的观察：
- Solar 日内周期很强，`Hour_of_Day` 的周期解释度约 0.575；`Month × Hour` 约 0.846。
- Thermal 的星期周期及 `Hour × DayOfWeek` 结构较明显，后者周期解释度约 0.221。
- Wind 的日内周期较弱（截图中约 0.002），但其长期自相关较强。
这些结果应在当前数据重新运行后核对。

## 5. Autocorrelation (ACF): hourly persistence and periodicity

In [ ]:
acf_sources = [c for c in ["Thermal", "Wind", "Solar", "Hydro", "Nuclear", "PumpedStorage"] if c in hourly.columns]
fig, axes = plt.subplots(len(acf_sources), 1, figsize=(13, 2.5 * len(acf_sources)))
if len(acf_sources) == 1:
    axes = [axes]

for ax, source in zip(axes, acf_sources):
    x = hourly[source]
    # Conservative missing-value handling preserves the original hourly lag positions.
    # Use up to one week of hourly lags to keep the plot readable.
    plot_acf(x, lags=168, zero=False, missing="conservative",
             ax=ax, title=f"{source}: ACF (1–168 hours)")
    ax.set_xlabel("Lag (hours)")

plt.tight_layout()
plt.show()

In [ ]:
# Selected autocorrelation values at economically meaningful lags.
acf_rows = []
for source in acf_sources:
    x = hourly[source]
    # Pair observations by exact hourly lag; do not compress missing timestamps.
    acf_rows.append({
        "Source": source,
        "ACF_1h": x.corr(x.shift(1)),
        "ACF_24h": x.corr(x.shift(24)),
        "ACF_168h": x.corr(x.shift(168)),
        "ACF_8760h": x.corr(x.shift(8760)) if len(x) > 8760 else np.nan
    })

acf_table = pd.DataFrame(acf_rows).set_index("Source")
display(acf_table.round(3))

### How to interpret ACF

- **Lag 1h**：相邻小时是否高度连续，反映短期惯性。
- **Lag 24h**：昨天同一小时与今天同一小时是否相似，提示日内周期。
- **Lag 168h**：上周同一小时与本周是否相似，提示周周期。
- **Lag 8760h**：去年同一小时与今年是否相似，提示年度周期（数据至少覆盖一年以上才有意义）。

ACF 高不等于一定存在季节性：趋势也会造成高自相关。应结合时间序列图和周期均值图一起判断。

## 6. Pairwise correlation between generation sources

In [ ]:
corr_data = hourly.copy()
corr = corr_data.corr(method="pearson")

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            vmin=-1, vmax=1, square=True, linewidths=0.5)
plt.title("Pearson correlation between hourly generation sources")
plt.tight_layout()
plt.show()

display(corr.round(3))

### Correlation interpretation

相关系数描述同期线性共变关系，不等同于因果关系。尤其要注意：

- Wind 与 Solar 都属于可再生供给，但两者可能因天气和季节不同而不呈正相关。
- Thermal 与可再生能源负相关，与 merit-order displacement 的经济直觉一致，但同期相关还可能受到需求、进口、核电、水电等遗漏因素影响。
- 如果序列存在趋势或季节性，原始水平相关可能被共同周期影响；可进一步对去季节化残差计算相关，作为稳健性补充，而不是直接把原始相关解释为结构性替代弹性。

此前截图中 Wind 与 Thermal 的 Pearson 相关约 -0.323，Solar 与 Thermal 约 -0.114，Wind + Solar 与 Thermal 约 -0.360。以本次实际运行结果为准。

## 7. Extra plot 1 — Solar and Thermal Month × Hour seasonal heatmaps

In [ ]:
# Convert to German local time before extracting calendar fields.
local = hourly.copy()
local.index = local.index.tz_convert("Europe/Berlin")
local["Hour"] = local.index.hour
local["Month"] = local.index.month

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, source in zip(axes, ["Solar", "Thermal"]):
    if source not in local.columns:
        ax.set_visible(False)
        continue
    profile = local.pivot_table(index="Month", columns="Hour", values=source, aggfunc="mean")
    sns.heatmap(profile, cmap="YlOrRd", ax=ax, cbar_kws={"label": "Mean MW"})
    ax.set_title(f"{source}: average generation by Month × Hour")
    ax.set_xlabel("Hour of day (Germany local time)")
    ax.set_ylabel("Month")

plt.tight_layout()
plt.show()

**图的含义：** 这张图把“日内周期”和“年度季节性”放在一起看。Solar 应呈现明显的白天高、夜间低，并且夏季白昼发电更强；Thermal 的日内形状则可能随月份变化，反映负荷和新能源供给共同作用。它能直观验证为什么 Solar 的季节性不能只用一个简单的 Month 或 Hour 特征概括。

## 8. Extra plot 2 — Thermal generation across Wind + Solar supply regimes

In [ ]:
# Use Wind + Solar as the renewable supply proxy, consistent with Part 2's input definition.
if {"Thermal", "Wind", "Solar"}.issubset(hourly.columns):
    regime = hourly[["Thermal", "Wind", "Solar"]].dropna().copy()
    regime["Wind_Solar"] = regime["Wind"] + regime["Solar"]
    regime["Renewable_Quintile"] = pd.qcut(
        regime["Wind_Solar"].rank(method="first"),
        q=5, labels=["Q1 Low", "Q2", "Q3", "Q4", "Q5 High"]
    )

    regime_summary = regime.groupby("Renewable_Quintile", observed=False).agg(
        Mean_Wind_Solar_MW=("Wind_Solar", "mean"),
        Mean_Thermal_MW=("Thermal", "mean"),
        Median_Thermal_MW=("Thermal", "median"),
        Std_Thermal_MW=("Thermal", "std"),
        Observations=("Thermal", "size")
    )

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(regime_summary.index.astype(str), regime_summary["Mean_Thermal_MW"],
            marker="o", lw=2, label="Mean Thermal")
    ax.fill_between(
        np.arange(len(regime_summary)),
        regime_summary["Mean_Thermal_MW"] - regime_summary["Std_Thermal_MW"],
        regime_summary["Mean_Thermal_MW"] + regime_summary["Std_Thermal_MW"],
        alpha=0.18, label="±1 SD"
    )
    ax.set_xticks(np.arange(len(regime_summary)))
    ax.set_xticklabels(regime_summary.index.astype(str))
    ax.set_xlabel("Wind + Solar generation quintile")
    ax.set_ylabel("Thermal generation (MW)")
    ax.set_title("Thermal generation across renewable supply regimes")
    ax.legend()
    plt.tight_layout()
    plt.show()

    display(regime_summary.round(1))
else:
    print("Thermal, Wind and Solar are required for this plot.")

**图的含义：** 将 Wind + Solar 按供给水平分成五组，观察 Thermal 平均出力是否随可再生供给增加而下降，以及下降是否呈线性。此前截图显示 Thermal 均值从最低新能源组约 5,975 MW 降至最高组约 4,117 MW，方向上符合替代关系；局部斜率并非完全恒定，提示可能存在非线性。该图是分组描述，不是控制需求等因素后的因果估计。

## 9. Compact findings checklist for Question 1

完成运行后，建议按以下框架写答案：

1. **Trend：** 指出主要能源的长期水平变化和明显阶段性变化，并结合时间序列图描述。
2. **Seasonality：** 用 ACF（24h / 168h / 8760h）、Month × Hour 热力图说明日内、周内和年度周期。Solar 的日内与季节交互最突出；Thermal 还体现一定星期及日内结构。
3. **Cross-source relationship：** 用相关热力图报告 Thermal 与 Wind、Solar 的同期关系，并明确相关不代表因果。
4. **Merit-order pattern：** 用 Wind + Solar 五分位图说明新能源供给越高时 Thermal 是否倾向下降，并观察是否存在非线性。
5. **Data issues：** 报告缺失率、时间连续性、负值和异常值；说明聚合时部分子类缺失可能低估该类别总量。

**数据解释边界：** 题目提供的是按技术类别的发电量，不包含完整电力需求、跨境净进口和所有系统约束。因此不能仅凭这些图断言 Thermal 完全“补足”新能源缺口；应表述为观察到与 merit-order displacement 相一致的统计关系。